In [ ]:
import pandas as pd

df = pd.read_csv("app_user_behavior_dataset.csv")

df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df["rating_given"] = df["rating_given"].fillna(df["rating_given"].median())

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.hist(df["age"], bins=20)
plt.xlabel("Age")
plt.ylabel("Number of Users")
plt.title("Age Distribution of App Users")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["sessions_per_week"], bins=15)
plt.xlabel("Sessions per Week")
plt.ylabel("Number of Users")
plt.title("Weekly Session Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(df["daily_active_minutes"], bins=20)

plt.xlabel("Daily Active Minutes")
plt.ylabel("Number of Users")
plt.title("Daily Active Minutes Distribution")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(df["engagement_score"], bins=20)

plt.xlabel("Engagement Score")
plt.ylabel("Number of Users")
plt.title("Engagement Score Distribution")

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(df["churn_risk_score"], bins=20)

plt.xlabel("Churn Risk Score")
plt.ylabel("Number of Users")
plt.title("Churn Risk Score Distribution")

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

numeric_df = df.select_dtypes(include="number").drop(columns=["user_id"])

plt.figure(figsize=(16, 12))

sns.heatmap(
    numeric_df.corr(),
    annot=False,
    cmap="coolwarm",
    linewidths=0.5
)

plt.title("Correlation Heatmap of Numerical Features")
plt.show()

In [ ]:
features = [
    "sessions_per_week",
    "avg_session_duration_min",
    "daily_active_minutes",
    "feature_clicks_per_session",
    "notifications_opened_per_week",
    "in_app_search_count",
    "pages_viewed_per_session",
    "crash_events_last_30_days",
    "support_tickets_raised",
    "days_since_last_login",
    "ads_clicked_last_30_days",
    "content_downloads",
    "social_shares",
    "rating_given",
    "churn_risk_score",
    "engagement_score"
]

X = df[features]

X.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled[:5]

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

inertia = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), inertia, marker="o")

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal K")

plt.show()

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_scaled)

In [ ]:
df["Cluster"] = clusters

In [ ]:
df["Cluster"].value_counts().sort_index()

In [ ]:
cluster_profile = df.groupby("Cluster")[features].mean().round(2)

cluster_profile


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 6))

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df["Cluster"],
    s=10
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("User Behavior Clusters using PCA")

plt.show()

In [ ]:
cluster_summary = df.groupby("Cluster")[
    ["sessions_per_week",
     "daily_active_minutes",
     "engagement_score",
     "churn_risk_score",
     "rating_given",
     "days_since_last_login"]
].mean().round(2)

cluster_summary

In [ ]:
import matplotlib.pyplot as plt

cluster_counts = df["Cluster"].value_counts().sort_index()

plt.figure(figsize=(8, 5))
plt.bar(cluster_counts.index.astype(str), cluster_counts.values)

plt.xlabel("Cluster")
plt.ylabel("Number of Users")
plt.title("Number of Users in Each Cluster")

plt.show()

In [ ]:
cluster_labels = {
    0: "Cluster 0",
    1: "Cluster 1",
    2: "Cluster 2",
    3: "Cluster 3"
}

df["Segment"] = df["Cluster"].map(cluster_labels)

In [ ]:
df[["Cluster", "Segment"]].head()

In [ ]:
df.to_csv("app_user_behavior_clustered.csv", index=False)

print("Clustered dataset saved successfully!")

In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

# Page configuration
st.set_page_config(
    page_title="App User Behavior Segmentation",
    page_icon="📊",
    layout="wide"
)

# Load dataset
df = pd.read_csv("app_user_behavior_clustered.csv")

# Title
st.title("📊 App User Behavior Segmentation")

st.write(
    "This dashboard analyzes app user behavior and displays "
    "user segments created using K-Means clustering."
)

# -----------------------------
# Key Metrics
# -----------------------------

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric("Total Users", f"{len(df):,}")

with col2:
    st.metric("Number of Clusters", df["Cluster"].nunique())

with col3:
    st.metric(
        "Average Engagement Score",
        round(df["engagement_score"].mean(), 2)
    )

with col4:
    st.metric(
        "Average Churn Risk",
        round(df["churn_risk_score"].mean(), 2)
    )

# -----------------------------
# Cluster Distribution
# -----------------------------

st.subheader("👥 User Distribution by Cluster")

cluster_counts = df["Cluster"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(
    cluster_counts.index.astype(str),
    cluster_counts.values
)

ax.set_xlabel("Cluster")
ax.set_ylabel("Number of Users")
ax.set_title("Number of Users in Each Cluster")

st.pyplot(fig)

# -----------------------------
# Cluster Profile
# -----------------------------

st.subheader("📋 Cluster Profile")

profile_columns = [
    "sessions_per_week",
    "daily_active_minutes",
    "engagement_score",
    "churn_risk_score",
    "rating_given",
    "days_since_last_login"
]

cluster_profile = (
    df.groupby("Cluster")[profile_columns]
    .mean()
    .round(2)
)

st.dataframe(cluster_profile)

# -----------------------------
# User Data Preview
# -----------------------------

st.subheader("🔍 User Data")

st.dataframe(df.head(100))